In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

print(os.listdir('/kaggle/input'))

In [ ]:
import os

print(os.listdir('/kaggle/input/datasets'))

In [ ]:
import os

print(os.listdir('/kaggle/input/datasets/puneet6060'))

In [ ]:
print(os.listdir('/kaggle/input/datasets/puneet6060/intel-image-classification'))

In [ ]:
train_path = '/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train/seg_train'

print(os.listdir(train_path))

In [ ]:
from keras.utils import image_dataset_from_directory

train_path = '/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train/seg_train'
test_path = '/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test'

train_ds = image_dataset_from_directory(
    train_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=True
)

test_ds = image_dataset_from_directory(
    test_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

In [ ]:
print(train_ds.class_names)
print(train_ds.cardinality())
print(test_ds.cardinality())

In [ ]:
from keras.applications.mobilenet_v2 import preprocess_input

In [ ]:
def preprocess(image, label):
    image = preprocess_input(image)
    return image, label

In [ ]:
train_ds = train_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

In [ ]:
from keras.applications import MobileNetV2

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [ ]:
base_model.summary()

In [ ]:
base_model.trainable = False

In [ ]:
print("Trainable parameters:", sum(
    layer.count_params()
    for layer in base_model.layers
    if layer.trainable
))

In [ ]:
from keras.models import Sequential
from keras.layers import GlobalAveragePooling2D, Dense, Dropout

model = Sequential()

model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(6, activation='softmax'))

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print("Test accuracy:", accuracy)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Training', 'Validation'])
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Training', 'Validation'])
plt.show()

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [ ]:
from keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_fine = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print("Fine-tuned accuracy:", accuracy)

In [ ]:
import numpy as np

y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
class_names = [
    'buildings',
    'forest',
    'glacier',
    'mountain',
    'sea',
    'street'
]

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))


In [ ]:
from keras.applications import ResNet50

base_model=ResNet50(
    weights = 'imagenet',
    include_top = False,
    input_shape=(224,224,3)
)

In [ ]:
base_model.trainable = False;

In [ ]:
from keras.applications.resnet50 import preprocess_input

In [ ]:
def preprocess_resnet(image,label) : 
    image = preprocess_input(image)
    return image,label

In [ ]:
train_ds = train_ds.map(preprocess_resnet)
test_ds = test_ds.map(preprocess_resnet)

In [ ]:
from keras.utils import image_dataset_from_directory

In [ ]:
train_ds = image_dataset_from_directory(
    train_path,
    image_size=(224,224),
    batch_size = 32,
    shuffle = True
    
)
test_ds = image_dataset_from_directory(
    test_path,
    image_size = (224,224),
    batch_size = 32,
    shuffle = False
)

In [ ]:
class_names = train_ds.class_names

In [ ]:
from keras.models import Sequential
from keras.layers import GlobalAveragePooling2D,Dense,Dropout

In [ ]:
model = Sequential()

model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dense(128,activation= 'relu'))
model.add(Dropout(0.3))
model.add(Dense(6,activation='softmax'))

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

In [ ]:
history = model.fit(
    test_ds,
    epochs = 10,
    validation_data = test_ds
)

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print("ResNet50 accuracy:", accuracy)

In [ ]:
print(train_ds.element_spec)

In [ ]:
for images, labels in train_ds.take(1):
    print(images.numpy().min())
    print(images.numpy().max())

#  resnet preprocessing has not applied 

In [ ]:
from keras.utils import image_dataset_from_directory

train_ds = image_dataset_from_directory(
    train_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=True
)

test_ds = image_dataset_from_directory(
    test_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

class_names = train_ds.class_names

In [ ]:
from keras.applications.resnet50 import preprocess_input
def preprocess_resnet(image, label):
    image = preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess_resnet)
test_ds = test_ds.map(preprocess_resnet)

In [ ]:
for images, labels in train_ds.take(1):
    print("Min:", images.numpy().min())
    print("Max:", images.numpy().max())

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

In [ ]:
model = Sequential()

model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(6, activation='softmax'))

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

In [ ]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    callbacks=[early_stop]
)